# Notebook 5 — Claim Risk Scoring and LangChain Summary

This notebook fuses the visual and acoustic reports into a **human-review triage** result. The score is deterministic and auditable; LangChain + a local Ollama model produces a structured narrative from those computed facts only. It is not used to make approval, payment, liability, or fraud decisions.

In [ ]:
# Run once in a fresh environment if needed
# !pip install langchain-core langchain-ollama pandas --quiet

import json
from pathlib import Path
import pandas as pd

from claim_analyzer import compute_claim_risk, generate_claim_summary

ROOT = Path.cwd()
VISUAL_REPORT_PATH = ROOT / 'all_videos_visual_reports.json'
AUDIO_REPORT_PATH = ROOT / 'all_videos_audio_reports.json'
OUTPUT_DIR = ROOT / 'reports'
OUTPUT_DIR.mkdir(exist_ok=True)

with open(VISUAL_REPORT_PATH, encoding='utf-8') as file:
    visual_reports = json.load(file)
with open(AUDIO_REPORT_PATH, encoding='utf-8') as file:
    audio_reports = json.load(file)

common_videos = sorted(set(visual_reports) & set(audio_reports))
print(f'Visual reports: {len(visual_reports)} | Audio reports: {len(audio_reports)} | Paired reports: {len(common_videos)}')
assert common_videos, 'No matching video names were found in the two report files.'

## Scoring policy

**Claim Risk Score = 80% Visual Damage Score + 20% Acoustic Stress Score.** Visual evidence has the larger weight because it is directly relevant to visible vehicle damage. Acoustic stress is context only and must never be used by itself for a fraud, coverage, payment, or liability decision. If audio is unavailable, visual evidence is weighted at 100%.

Risk bands: low < 35, moderate 35–64.9, high ≥ 65. The recommendation is always a human-review queue priority.

In [ ]:
# Set True after `ollama serve` and `ollama pull llama3.2:3b`.
# The chain is PromptTemplate -> ChatOllama -> StrOutputParser -> JSON validation.
USE_OLLAMA = True

claim_reports = {}
for video_name in common_videos:
    visual = visual_reports[video_name]
    audio = audio_reports[video_name]
    risk = compute_claim_risk(visual, audio)
    summary = generate_claim_summary(visual, audio, risk, use_ollama=USE_OLLAMA)
    claim_reports[video_name] = {
        'visual_report': visual,
        'audio_report': audio,
        'risk_assessment': risk,
        'claim_summary': summary,
    }

summary_table = pd.DataFrame([
    {
        'video': name,
        'risk_score': report['risk_assessment']['claim_risk_score'],
        'risk_band': report['risk_assessment']['risk_band'],
        'visual_score': report['risk_assessment']['visual_damage_score'],
        'acoustic_score': report['risk_assessment']['acoustic_stress_score'],
        'signal_relationship': report['risk_assessment']['signal_relationship'],
        'summary_source': report['claim_summary']['summary_source'],
    }
    for name, report in claim_reports.items()
])
summary_table.sort_values('risk_score', ascending=False).reset_index(drop=True)

In [ ]:
with open(OUTPUT_DIR / 'all_claim_risk_reports.json', 'w', encoding='utf-8') as file:
    json.dump(claim_reports, file, indent=2)

# Demo artifacts: one high priority, one low priority, and one cross-modal conflict.
ordered = summary_table.sort_values('risk_score')
low_name = ordered.iloc[0]['video']
high_name = ordered.iloc[-1]['video']
conflicts = summary_table[summary_table['signal_relationship'] == 'conflict']
conflict_name = conflicts.iloc[0]['video'] if not conflicts.empty else None
demo_names = {'low_risk': low_name, 'high_risk': high_name}
if conflict_name:
    demo_names['conflicting_signals'] = conflict_name

examples = {label: claim_reports[name] | {'video_name': name} for label, name in demo_names.items()}
with open(OUTPUT_DIR / 'demo_claim_examples.json', 'w', encoding='utf-8') as file:
    json.dump(examples, file, indent=2)

print('Saved:', OUTPUT_DIR / 'all_claim_risk_reports.json')
print('Saved:', OUTPUT_DIR / 'demo_claim_examples.json')
for label, name in demo_names.items():
    report = claim_reports[name]
    print(f'\n{label}: {name}')
    print(json.dumps(report['risk_assessment'], indent=2))
    print(report['claim_summary']['narrative'])